# Inferência para dados numéricos

[![Abrir no Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gustavocxavier/mqcc/blob/main/codigos/unidade1-topico2-3-inferencia-numericos.ipynb)

## Nascimentos na Carolina do Norte

Em 2004, o estado da Carolina do Norte divulgou um grande conjunto de dados com informações sobre os nascimentos registrados no estado. Esse conjunto de dados é útil para pesquisadores que estudam a relação entre os hábitos e as práticas de gestantes e o nascimento de seus filhos. Trabalharemos com uma amostra aleatória de observações desse conjunto de dados.

## Análise exploratória

Carregue o conjunto de dados `nc` em nosso notebook.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import io
import requests

df_url = 'https://raw.githubusercontent.com/akmand/datasets/master/openintro/nc.csv'
url_content = requests.get(df_url, verify=False).content
nc = pd.read_csv(io.StringIO(url_content.decode('utf-8')))

Temos observações de 13 variáveis diferentes, algumas categóricas e outras numéricas. O significado de cada variável é apresentado a seguir.

| variável         | descrição |
| ---------------- | ------------|
| `fage`           | idade do pai, em anos. |
| `mage`           | idade da mãe, em anos. |
| `mature`         | classificação da mãe quanto à maturidade. |
| `weeks`          | duração da gestação, em semanas. |
| `premie`         | se o nascimento foi classificado como prematuro (`premie`) ou a termo. |
| `visits`         | número de consultas hospitalares durante a gestação. |
| `marital`        | se a mãe estava `married` ou `not married` no momento do nascimento. |
| `gained`         | peso ganho pela mãe durante a gestação, em libras. |
| `weight`         | peso do bebê ao nascer, em libras. |
| `lowbirthweight` | se o bebê foi classificado com baixo peso ao nascer (`low`) ou não (`not low`). |
| `gender`         | sexo do bebê, `female` ou `male`. |
| `habit`          | condição da mãe como `nonsmoker` ou `smoker`. |
| `whitemom`       | se a mãe é `white` ou `not white`. |

<div class = 'exercise'>
<h4>Exercício 1</h4>
Quais são os casos deste conjunto de dados? Quantos casos existem em nossa amostra?
</div>

Como primeira etapa da análise, devemos examinar resumos dos dados. Isso pode ser feito usando `describe()` e `info()`:

In [2]:
nc.describe()

,fage,mage,weeks,visits,gained,weight
count,829.000000,1000.000000,998.000000,991.000000,973.000000,1000.00000
mean,30.255730,27.000000,38.334669,12.104945,30.325797,7.10100
std,6.763766,6.213583,2.931553,3.954934,14.241297,1.50886
min,14.000000,13.000000,20.000000,0.000000,0.000000,1.00000
25%,25.000000,22.000000,37.000000,10.000000,20.000000,6.38000
50%,30.000000,27.000000,39.000000,12.000000,30.000000,7.31000
75%,35.000000,32.000000,40.000000,15.000000,38.000000,8.06000
max,55.000000,50.000000,45.000000,30.000000,85.000000,11.75000


In [3]:
nc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   fage            829 non-null    float64
 1   mage            1000 non-null   int64  
 2   mature          1000 non-null   object 
 3   weeks           998 non-null    float64
 4   premie          998 non-null    object 
 5   visits          991 non-null    float64
 6   marital         999 non-null    object 
 7   gained          973 non-null    float64
 8   weight          1000 non-null   float64
 9   lowbirthweight  1000 non-null   object 
 10  gender          1000 non-null   object 
 11  habit           999 non-null    object 
 12  whitemom        998 non-null    object 
dtypes: float64(5), int64(1), object(7)
memory usage: 101.7+ KB


Ao examinar os resumos das variáveis, considere quais delas são categóricas e quais são numéricas. Entre as variáveis numéricas, há valores atípicos? Se você não tiver certeza ou quiser examinar os dados mais detalhadamente, construa um gráfico.

Considere a possível relação entre o hábito de fumar da mãe e o peso do bebê. Representar os dados graficamente é uma primeira etapa útil, pois nos ajuda a visualizar tendências rapidamente, identificar associações fortes e formular questões de pesquisa.

<div class = 'exercise'>
<h4>Exercício 2</h4>
    Construa boxplots lado a lado de <code>habit</code> e <code>weight</code>. O que o gráfico destaca sobre a relação entre essas duas variáveis?
</div>

Os boxplots mostram como as medianas das duas distribuições se comparam, mas também podemos comparar as médias das distribuições usando a função a seguir para separar a variável `weight` nos grupos de `habit` e, depois, calcular a média de cada grupo usando `mean()`.

In [4]:
nc.groupby(['habit'])['weight'].mean()

habit
nonsmoker    7.144273
smoker       6.828730
Name: weight, dtype: float64

Há uma diferença observada, mas ela é estatisticamente significativa? Para responder a essa pergunta, realizaremos um teste de hipóteses.

## Inferência

<div class = "exercise">
<h4>Exercício 3</h4>
Verifique se as condições necessárias para a inferência são atendidas. Observe que será preciso obter os tamanhos das amostras para verificar as condições. Você pode calcular o tamanho dos grupos usando o mesmo comando <code>groupby</code> acima, mas substituindo <code>mean</code> por <code>size</code>.
</div>

<div class = "exercise">
<h4>Exercício 4</h4>
Escreva as hipóteses para testar se os pesos médios dos bebês nascidos de mães fumantes e não fumantes são diferentes.
</div>

Agora realizaremos testes de hipóteses para verificar se os pesos médios dos bebês nascidos de mães fumantes e não fumantes são diferentes. Para essa tarefa, podemos usar [`statsmodels`](https://www.statsmodels.org/stable/index.html), um módulo Python que oferece classes e funções para estimar diversos modelos estatísticos, realizar testes estatísticos e explorar dados estatisticamente.

In [5]:
import statsmodels.stats.weightstats as st

nc_weightANDsmoker = nc[nc['habit'] == 'smoker']['weight']
nc_weightANDnonsmoker = nc[nc['habit'] == 'nonsmoker']['weight']

dsw1 = st.DescrStatsW(nc_weightANDsmoker)
dsw2 = st.DescrStatsW(nc_weightANDnonsmoker)
cm = st.CompareMeans(dsw1, dsw2)

# calcula o número de observações, a média e o desvio-padrão de cada grupo
n_smoker = dsw1.nobs
n_nonsmoker = dsw2.nobs
mean_smoker = dsw1.mean
mean_nonsmoker = dsw2.mean
sd_smoker = dsw1.std
sd_nonsmoker = dsw2.std
print(f'n_smoker = {n_smoker}')
print(f'mean_smoker = {mean_smoker}')
print(f'sd_smoker = {sd_smoker}')
print()
print(f'n_nonsmoker = {n_nonsmoker}')
print(f'mean_nonsmoker = {mean_nonsmoker}')
print(f'sd_nonsmoker = {sd_nonsmoker}')
print()

# realiza o teste de hipóteses
ht = cm.ztest_ind(alternative = 'two-sided', usevar = 'unequal', value = 0)

# calcula e exibe o erro-padrão, o escore Z e o valor-p do teste de hipóteses
se = cm.std_meandiff_separatevar
testZ = ht[0]
p_value = ht[1]
print(f'Erro-padrão = {se}')
print(f'Estatística de teste: Z = {testZ}')
print(f'valor-p = {p_value}')

# rejeita ou aceita a hipótese nula
if(p_value) < 0.05:
    print('rejeitar a hipótese nula')
else:
    print('aceitar a hipótese nula')

n_smoker = 126.0
mean_smoker = 6.828730158730157
sd_smoker = 1.380668106117173

n_nonsmoker = 873.0
mean_nonsmoker = 7.144272623138621
sd_nonsmoker = 1.5178105512705897

Standard error = 0.13376049190705977
Test statistic: Z = -2.359010944933654
p-value = 0.018323715325158647
reject null hypothesis


<div class = 'exercise'>
<h4>Exercício 5</h4>
Construa um intervalo de confiança para a diferença entre os pesos dos bebês nascidos de mães fumantes e não fumantes.</div>

---
## Agora é com você

<ol>
    <li>Calcule um intervalo de confiança de 95% para a duração média das gestações (<code>weeks</code>) e interprete-o no contexto. Observe que, como você está realizando inferência sobre um único parâmetro populacional, não há variável explicativa; portanto, pode omitir a variável <code>x</code> da função.</li><br>
    <li>Calcule um novo intervalo de confiança para o mesmo parâmetro, agora com nível de confiança de 90%.</li><br>
    <li>Realize um teste de hipóteses para avaliar se o peso médio ganho pelas mães mais jovens é diferente do peso médio ganho pelas mães maduras.</li><br>
    <li>Agora, uma tarefa sem inferência: determine o ponto de corte de idade entre mães mais jovens e maduras. Use um método de sua escolha e explique como ele funciona.</li><br>
    <li>Escolha um par formado por uma variável numérica e uma variável categórica e formule uma questão de pesquisa que avalie a relação entre elas. Formule a questão de modo que possa ser respondida por meio de um teste de hipóteses e/ou de um intervalo de confiança. Responda à questão com Python, informe os resultados estatísticos e também apresente uma explicação em linguagem simples.</li>
</ol>

<div class = "license">
Este laboratório foi adaptado por David Akman e Imran Ture a partir do OpenIntro, de Andrew Bray e Mine Çetinkaya-Rundel.
</div>

***

[Acesse o notebook original no GitHub](https://github.com/akmand/statististics_tutorials/blob/main/ch7_inf_for_numerical_data.ipynb)